<div style="background: linear-gradient(135deg, #0A3D62 0%, #3C6382 100%);
            color: white; padding: 26px 30px; border-radius: 14px;
            font-family: system-ui, sans-serif; margin-bottom: 6px; box-shadow: 0 4px 14px rgba(10,61,98,0.25);">
  <h1 style="margin:0; font-size:1.85rem; font-weight:700;">📊 Titanic — Baseline Models</h1>
  <p style="margin:10px 0 0; opacity:0.92;">
    Gender baseline → LR → RF → HistGBM · stratified CV · segmented error analysis (§60.21) · submissions
  </p>
</div>


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;"><h2 style="margin:0; font-size:1.25rem;">1. Setup &amp; Load</h2></div>


In [ ]:
from __future__ import annotations
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"axes.titlecolor": "#0A3D62"})

DATA_DIR = Path("../Datasets")
RANDOM_STATE = 42
N_SPLITS = 5
np.random.seed(RANDOM_STATE)

train_fe = pd.read_csv(DATA_DIR / "train_fe.csv")
test_fe = pd.read_csv(DATA_DIR / "test_fe.csv")
y = train_fe["Survived"].astype(int)
pid_test = test_fe["PassengerId"].copy()

assert train_fe["PassengerId"].is_unique
assert y.isin([0, 1]).all()
print(f"Train FE {train_fe.shape} | positive rate {y.mean():.3f}")


In [ ]:
NUM = ["Pclass","Age","SibSp","Parch","Fare","LogFare","FamilySize","IsAlone","TicketGroupSize","HasCabin","SexMale"]
CAT = ["Sex","Embarked","Title","Deck","FamilyBin","AgeBin","FareBin"]
for c in CAT:
    train_fe[c] = train_fe[c].astype(str)
    test_fe[c] = test_fe[c].astype(str)
X_train = train_fe[NUM + CAT].copy()
X_test = test_fe[NUM + CAT].copy()


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;"><h2 style="margin:0; font-size:1.25rem;">2. CV Helper (§61.16)</h2></div>


In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
SCORING = {"accuracy": "accuracy", "roc_auc": "roc_auc", "f1": "f1"}

def evaluate(name, est, X, y):
    r = cross_validate(est, X, y, cv=cv, scoring=SCORING, n_jobs=-1)
    row = {
        "model": name,
        "accuracy_mean": r["test_accuracy"].mean(),
        "accuracy_std": r["test_accuracy"].std(),
        "roc_auc_mean": r["test_roc_auc"].mean(),
        "roc_auc_std": r["test_roc_auc"].std(),
        "f1_mean": r["test_f1"].mean(),
        "f1_std": r["test_f1"].std(),
    }
    print(f"{name:28s} acc={row['accuracy_mean']:.4f}±{row['accuracy_std']:.4f}  "
          f"auc={row['roc_auc_mean']:.4f}  f1={row['f1_mean']:.4f}")
    return pd.Series(row)


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;"><h2 style="margin:0; font-size:1.25rem;">3. Models</h2></div>


In [ ]:
class GenderBaseline(BaseEstimator, ClassifierMixin):
    def fit(self, X, y=None):
        self.classes_ = np.array([0, 1])
        return self
    def predict(self, X):
        sex = X["Sex"] if isinstance(X, pd.DataFrame) else X[:, 0]
        return (pd.Series(sex).astype(str).str.lower().values == "female").astype(int)
    def predict_proba(self, X):
        p = self.predict(X)
        return np.column_stack([1 - p, p]).astype(float)

gender_clf = GenderBaseline()
gender_row = evaluate("Gender baseline", gender_clf, train_fe[["Sex"]], y)

prep_lr = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CAT),
])
lr_pipe = Pipeline([("prep", prep_lr), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
lr_row = evaluate("Logistic Regression", lr_pipe, X_train, y)

prep_tree = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUM),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CAT),
])
rf_pipe = Pipeline([("prep", prep_tree),
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=4,
                                   random_state=RANDOM_STATE, n_jobs=-1))])
rf_row = evaluate("Random Forest", rf_pipe, X_train, y)

hgb_pipe = Pipeline([("prep", prep_tree),
    ("clf", HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=200,
                                           random_state=RANDOM_STATE))])
hgb_row = evaluate("Hist Gradient Boosting", hgb_pipe, X_train, y)


In [ ]:
comparison = pd.DataFrame([gender_row, lr_row, rf_row, hgb_row]).sort_values("accuracy_mean", ascending=False)
display(comparison)

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(comparison))
w = 0.28
ax.bar(x - w, comparison["accuracy_mean"], w, yerr=comparison["accuracy_std"], label="Accuracy", color="#0A3D62", capsize=4)
ax.bar(x, comparison["roc_auc_mean"], w, yerr=comparison["roc_auc_std"], label="ROC-AUC", color="#3C6382", capsize=4)
ax.bar(x + w, comparison["f1_mean"], w, yerr=comparison["f1_std"], label="F1", color="#27AE60", capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(comparison["model"], rotation=15, ha="right")
ax.set_ylim(0.5, 1.0)
ax.set_title("Stratified 5-fold CV comparison")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;">
  <h2 style="margin:0; font-size:1.25rem;">4. Segmented Evaluation (§60.21)</h2>
</div>
<div style="background:#E8F1F5; border-radius:12px; padding:14px 18px; font-family:system-ui,sans-serif; border:1px solid #C5D9E3;">
  <p style="margin:0; color:#1B262C;">Overall accuracy can hide failures. Check accuracy by Sex and Pclass on full-train predictions.</p>
</div>


In [ ]:
best_name = comparison.iloc[0]["model"]
zoo = {
    "Gender baseline": gender_clf,
    "Logistic Regression": lr_pipe,
    "Random Forest": rf_pipe,
    "Hist Gradient Boosting": hgb_pipe,
}
best = zoo[best_name]
if best_name == "Gender baseline":
    best.fit(train_fe[["Sex"]], y)
    pred_tr = best.predict(train_fe[["Sex"]])
else:
    best.fit(X_train, y)
    pred_tr = best.predict(X_train)

train_fe = train_fe.copy()
train_fe["_pred"] = pred_tr
train_fe["_correct"] = (train_fe["_pred"] == train_fe["Survived"]).astype(int)

print(f"Best model: {best_name}")
print("\nAccuracy by Sex:")
print(train_fe.groupby("Sex")["_correct"].mean())
print("\nAccuracy by Pclass:")
print(train_fe.groupby("Pclass")["_correct"].mean())
print("\nAccuracy by Sex × Pclass:")
print(train_fe.groupby(["Sex", "Pclass"])["_correct"].mean().unstack())

cm = confusion_matrix(y, pred_tr)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Pred Died", "Pred Survived"],
            yticklabels=["Actual Died", "Actual Survived"])
ax.set_title(f"Confusion matrix — {best_name} (train)")
plt.tight_layout(); plt.show()


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;"><h2 style="margin:0; font-size:1.25rem;">5. Submissions (§60.22)</h2></div>


In [ ]:
for name, est in zoo.items():
    if name == "Gender baseline":
        est.fit(train_fe[["Sex"]], y)
        pred = est.predict(test_fe[["Sex"]])
    else:
        est.fit(X_train, y)
        pred = est.predict(X_test)
    sub = pd.DataFrame({"PassengerId": pid_test, "Survived": pred.astype(int)})
    path = DATA_DIR / f"submission_{name.lower().replace(' ', '_')}.csv"
    sub.to_csv(path, index=False)
    print(f"{name:28s} → {path.name}  survived_rate={sub['Survived'].mean():.3f}")


<div style="background:#EAF7F0; border-left:5px solid #27AE60; border-radius:0 12px 12px 0; padding:16px 20px; font-family:system-ui,sans-serif;">
  <strong style="color:#1E8449;">Executive summary (§60.20)</strong>
  <ul style="margin:8px 0 0; padding-left:1.2rem; color:#1B262C; line-height:1.55;">
    <li>Gender baseline is a strong floor (~79% CV accuracy).</li>
    <li>Engineered features lift LR / RF / HistGBM to ~83%+ CV accuracy.</li>
    <li>Segment check: review accuracy by Sex and Pclass — do not ship on overall score alone.</li>
    <li>Associations support prediction; this is not a causal study of rescue policy.</li>
    <li>Artifacts: <code>Datasets/submission_*.csv</code> ready for leaderboard comparison.</li>
  </ul>
</div>
